In [8]:
from langgraph.graph import StateGraph , START ,END
from langchain_groq import ChatGroq 
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from typing import Annotated , TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage , AIMessage , BaseMessage
from langgraph.prebuilt import ToolNode , tools_condition

In [2]:
load_dotenv()


True

In [3]:
llm=ChatGroq(model='llama-3.3-70b-versatile')

In [4]:
loader=PyPDFLoader('intro-to-ml.pdf')
docs=loader.load()

In [5]:
len(docs)

392

In [10]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=splitter.split_documents(docs)

In [11]:
len(chunks)

973

In [14]:
embedding=HuggingFaceEmbeddings(model='BAAI/bge-base-en-v1.5')
vector_store=FAISS.from_documents(chunks , embedding)

d:\Clg Kabir_\Agentic-AI\myenv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8382.96it/s]


In [15]:
retriever=vector_store.as_retriever(search_type='similarity',search_kwargs={"k":4})


In [16]:
@tool
def rag_tool(query):
    """
    Retriever relevant information from the pdf documents.
    Use this tool when the user asks factual/conceptual questions
    that might be answered from the stored documents.
    """
    result=retriever.invoke(query)
    context=[docs.page_content for doc in result ]
    metadata=[docs.metadata for doc in result]

    return {
        "query":query,
        "context":context,
        "metadata":metadata
    }

In [17]:
tools=[rag_tool]
llm_with_tools=llm.bind_tools(tools)

In [18]:
class ChatState(StateGraph):
    messages:Annotated[list[BaseMessage],add_messages]


In [19]:
def chat_node(state: ChatState):
    messages=state['messages']
    response=llm_with_tools.invoke(messages)
    return {'messages':[response]}

In [20]:
tool_node=ToolNode(tools)


In [ ]:
graph=StateGraph(ChatState)

graph.add_node("Chat_Node",chat_node)
graph.add_node("tools",tool_node)

graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node",tools_condition)
graph.add_edge("tools","chat_node")

chatbot=graph.compile()